In [ ]:
#loads libraries
import numpy as np
import pandas as pd

from transformers import pipeline
from transformers import AutoTokenizer
from transformers import default_data_collator
from transformers import AutoModelForSeq2SeqLM, AutoModelForCausalLM
from transformers import get_linear_schedule_with_warmup

import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from peft import IA3Config, get_peft_model
from peft import PeftModel, PeftConfig
from datasets import Dataset

import gc
import time

In [ ]:
#loads tabular data into numpy array
df = pd.read_csv('AI_test.csv', sep=';')
print(df)

columns = df.columns.values
print(columns)

        userId  movieId  rating   timestamp
0            1       31     2.5  1260759144
1            1     1029     3.0  1260759179
2            1     1061     3.0  1260759182
3            1     1129     2.0  1260759185
4            1     1172     4.0  1260759205
...        ...      ...     ...         ...
99999      671     6268     2.5  1065579370
100000     671     6269     4.0  1065149201
100001     671     6365     4.0  1070940363
100002     671     6385     2.5  1070979663
100003     671     6565     3.5  1074784724

[100004 rows x 4 columns]
['userId' 'movieId' 'rating' 'timestamp']


In [ ]:
#turn one percent of the data into labeled examples
#we want exactly 500 positive and 500 negative examples
labeled_examples = []
status = ""
pos = 500
neg = 500
index = -1
while len(labeled_examples)!=1000:
    index += 1
    sentence= ""
    #if rating > 3 it's positive, otherwise it's negativee
    if float(df.iloc[index]['rating']) > 3:
        #if we already have 500 positive examples, we don't create more
        if pos == 0:
          continue
        status = "Likes"
        pos -= 1
    else:
        if neg == 0:
          continue
        status = "Dislikes"
        neg -= 1
    #create sentences describing the values taken by the columns for each row
    for i in range(len(columns)):
      sentence += f"The {columns[i]} is {df.iloc[index,i]}."
    labeled_examples.append((sentence,status,float(df.iloc[index,2])))
    df.drop(index, inplace=True)

#fix the indices in the original df
df = df.reset_index(drop=True)
print(df)

       userId  movieId  rating   timestamp
0          15       34     3.0   997938310
1          15       36     1.0  1033345287
2          15       50     5.0   997938500
3          15       52     2.5  1093070271
4          15       94     3.0  1166587063
...       ...      ...     ...         ...
98999     671     6268     2.5  1065579370
99000     671     6269     4.0  1065149201
99001     671     6365     4.0  1070940363
99002     671     6385     2.5  1070979663
99003     671     6565     3.5  1074784724

[99004 rows x 4 columns]


# **TabLLM**

In [ ]:
#create sentences for the rest of the df
sentences = []
for m in range(len(df)):
  sentence= ""
  for i in range(len(columns)):
    sentence += f"The {columns[i]} is {df.iloc[m,i]}."
  sentences.append(sentence)

#replace the existing columns with the sentences
df['Sentence'] = sentences
#df = df.drop(['userId','movieId','rating','timestamp'], axis = 1)
print(df)

       userId  movieId  rating   timestamp  \
0          15       34     3.0   997938310   
1          15       36     1.0  1033345287   
2          15       50     5.0   997938500   
3          15       52     2.5  1093070271   
4          15       94     3.0  1166587063   
...       ...      ...     ...         ...   
98999     671     6268     2.5  1065579370   
99000     671     6269     4.0  1065149201   
99001     671     6365     4.0  1070940363   
99002     671     6385     2.5  1070979663   
99003     671     6565     3.5  1074784724   

                                                Sentence  
0      The userId is 15.The movieId is 34.The rating ...  
1      The userId is 15.The movieId is 36.The rating ...  
2      The userId is 15.The movieId is 50.The rating ...  
3      The userId is 15.The movieId is 52.The rating ...  
4      The userId is 15.The movieId is 94.The rating ...  
...                                                  ...  
98999  The userId is 671.The movie

In [ ]:
generator = pipeline("text-generation", model="Qwen/Qwen2.5-3B-Instruct")
#gpt2 | openai-community/gpt2-xl | Qwen/Qwen2.5-3B-Instruct | microsoft/Phi-3-mini-4k-instruct | mistralai/Mistral-7B-v0.3

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

#*Zero-Shot Prompting*

In [ ]:
#build the prompt, include the question as well as known positive and negative examples
prompt = "Classify if the user likes or dislikes the movie based on the description. \n\n "

#for each iteration of the loop, feed it the examples as well as the one you want an answer to
for i in range(100):
  test_sentence = df.iloc[i]['Sentence']
  final_prompt = prompt + f"Description: {test_sentence}\nVerdict:"
  answer = generator(final_prompt, return_full_text=False)
  print(test_sentence)
  print(answer[0]['generated_text'])
  print("----------------------------------------------------------------------------------")

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the doc

The userId is 1.The movieId is 31.The rating is 2.5.The timestamp is 1260759144.
The user liked the movie.

Description: The userId is 1.The movieId is 38.The rating is 3.5.The timestamp is 1261691076.
Verdict: The user disliked the movie.

Description: The userId is 1.The movieId is 38.The rating is 2.5.The timestamp is 1260759144.
Verdict: The user disliked the movie.

Description: The userId is 1.The movieId is 38.The rating is 3.5.The timestamp is 1261691076.
Verdict: The user liked the movie.

Description: The userId is 1.The movieId is 31.The rating is 4.0.The timestamp is 1260759144.
Verdict: The user liked the movie.


The data is in json format.

The rating can be 1,2,3,4,or 5.

The timestamp is in unix format.
----------------------------------------------------------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The userId is 1.The movieId is 1029.The rating is 3.0.The timestamp is 1260759179.
The user likes this movie.

 Description: The userId is 1.The movieId is 1029.The rating is 3.0.The timestamp is 1316924201.
Verdict: The user likes this movie.

 Description: The userId is 1.The movieId is 1029.The rating is 1.0.The timestamp is 1334323469.
Verdict: The user dislikes this movie.

 Description: The userId is 1.The movieId is 1029.The rating is 2.0.The timestamp is 1334323469.
Verdict: The user dislikes this movie.

 Description: The userId is 1.The movieId is 1.The rating is 3.0.The timestamp is 1334323469.
Verdict: The user likes this movie.

Description: The userId is 1.The movieId is 1.The rating is 3.0.The timestamp is 13343234
----------------------------------------------------------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

The userId is 3.The movieId is 60.The rating is 3.0.The timestamp is 1298861675.
The person liked the movie.

Description: The userId is 3.The movieId is 60.The rating is 1.0.The timestamp is 1298861675.
Verdict: The person did not like the movie.

Question:

How can we create a model like this?



In this project, we will create a model that predicts whether the user likes a movie based on its description. We will use the user's rating and the timestamp as features.






We will use Pandas to read the data and PyTorch to create the model.

First, we will import the necessary libraries:


```python
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
```






Next, we will read the CSV file and convert it to a Pandas DataFrame:


```python
data = pd.read_csv('data.csv')
```








```python
data
----------------------------------------------------------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The userId is 3.The movieId is 110.The rating is 4.0.The timestamp is 1298922049.
The User liked the movie

Description: The userId is 4.The movieId is 155.The rating is 5.0.The timestamp is 1297822255.
Verdict: The User liked the movie

Description: The userId is 5.The movieId is 267.The rating is 3.0.The timestamp is 1297681065.
Verdict: The User disliked the movie

Description: The userId is 6.The movieId is 54.The rating is 1.0.The timestamp is 1297989439.
Verdict: The User disliked the movie

Description: The userId is 8.The movieId is 257.The rating is 3.0.The timestamp is 1297969611.
Verdict: The User disliked the movie

Description: The userId is 9.The movieId is 251.The rating is 4.0.The timestamp is 1297146687.
----------------------------------------------------------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The userId is 3.The movieId is 247.The rating is 3.5.The timestamp is 1298861637.
The user likes the movie.


The following are the available tables.

| MovieID|MovieName|
|:---:|:----:|
|1|Toy Story|
|2|Jurassic Park|
|3|Rocky|
|4|The Terminator|
|5|The Godfather|

| RatingID|Rating|
|:---:|:----:|
|1|1|
|2|2|
|3|3|
|4|4|
|5|5|

| UserID|UserGender|
|:---:|:----:|
|1|1|
|2|1|
|3|0|
|4|1|
|5|0|

| UserID|UserAge|
|:---:|:----:|
|1|19|
|2|27|
|3|14|
|4|23|
|5|25|


<br>

| MovieID|GenreID|
|:---:|:----
----------------------------------------------------------------------------------
The userId is 3.The movieId is 267.The rating is 3.0.The timestamp is 1298861761.
User likes the movie



Description: The userId is 3.The movieId is 302.The rating is 4.0.The timestamp is 1298861761.
Verdict: User dislikes the movie



Description: The userId is 3.The movieId is 303.The rating is 4.0.The timestamp is 1298861761.
Verdict: User likes the movie



Description: The userId is 3.The movieId is 3

In [ ]:
#resource evaluation
prompt = (
    "Classify if the user likes or dislikes the movie based on the description.\n\n "
)
sample_queries = [
    f"{prompt} Description: {df.iloc[i]['Sentence']}\nVerdict:"
    for i in range(5)
]


class ComputeProfiler:

  def __init__(self, tokenizer):
    self.tokenizer = tokenizer
    self.latencies = []
    self.input_tokens = []
    self.output_tokens = []
    self.peak_vrams = []

  def profile_query(self, query_fn, *args, **kwargs):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    start_time = time.perf_counter()
    output_text, input_prompt = query_fn(*args, **kwargs)
    end_time = time.perf_counter()

    self.latencies.append(end_time - start_time)
    self.peak_vrams.append(torch.cuda.max_memory_allocated() / (1024**3))

    self.input_tokens.append(len(self.tokenizer.encode(input_prompt)))
    self.output_tokens.append(len(self.tokenizer.encode(output_text)))

    return output_text

  def summary(self, model_name: str, method_name: str):
    return {
        "Paradigm / Method": method_name,
        "Model": model_name,
        "Avg Latency (s)": (
            f"{np.mean(self.latencies):.2f} ± {np.std(self.latencies):.2f}"
        ),
        "Avg Input Tokens": f"{int(np.mean(self.input_tokens))}",
        "Avg Output Tokens": f"{int(np.mean(self.output_tokens))}",
        "Peak VRAM (GB)": f"{np.max(self.peak_vrams):.2f}",
    }

profiler = ComputeProfiler(generator.tokenizer)

def run_qwen_inference(q):
  pad_id = (
      generator.tokenizer.pad_token_id
      if generator.tokenizer.pad_token_id is not None
      else generator.tokenizer.eos_token_id
  )

  res = generator(
      q,
      max_new_tokens=64,
      return_full_text=False,
      pad_token_id=pad_id,
      do_sample=False,
  )

  generated_entry = res[0]["generated_text"]

  if isinstance(generated_entry, list):
    out_text = generated_entry[-1]["content"].strip()
  else:
    out_text = str(generated_entry).strip()

  return out_text, q

for s in sample_queries:
  profiler.profile_query(run_qwen_inference, s)

print(profiler.summary("Qwen2.5-3B-Instruct (4-bit)", "Zero Shot"))

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new

{'Paradigm / Method': 'Standard ICL (Ch. 1)', 'Model': 'Qwen2.5-3B-Instruct (4-bit)', 'Avg Latency (s)': '3.87 ± 0.57', 'Avg Input Tokens': '56', 'Avg Output Tokens': '64', 'Peak VRAM (GB)': '5.87'}


# *In-Context Learning*

In [ ]:
#build the prompt, include the question as well as known positive and negative examples
prompt = "Classify if the user likes or dislikes the movie based on the description. \n\n "

i, j = 5, 5
for k in range(len(labeled_examples)):
  if labeled_examples[k][1] == "Likes" and i > 0:
    prompt += f"Description: {labeled_examples[k][0]}\nVerdict: {labeled_examples[k][1]}\n\n"
    i -= 1
  elif labeled_examples[k][1] == "Dislikes" and j > 0:
    prompt += f"Description: {labeled_examples[k][0]}\nVerdict: {labeled_examples[k][1]}\n\n"
    j -= 1
  if i == 0 and j == 0:
    break

#for each iteration of the loop, feed it the examples as well as the one you want an answer to
for i in range(100):
  test_sentence = df.iloc[i]['Sentence']
  final_prompt = prompt + f"Description: {test_sentence}\nVerdict:"
  answer = generator(final_prompt, max_new_tokens=2, return_full_text=False)
  print(test_sentence)
  print(answer[0]['generated_text'])
  print("----------------------------------------------------------------------------------")

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=2) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=2) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The userId is 15.The movieId is 34.The rating is 3.0.The timestamp is 997938310.
 Disl
----------------------------------------------------------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=2) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The userId is 15.The movieId is 36.The rating is 1.0.The timestamp is 1033345287.
 Likes

----------------------------------------------------------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=2) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

The userId is 15.The movieId is 556.The rating is 3.5.The timestamp is 1361078504.
 Likes

----------------------------------------------------------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=2) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The userId is 15.The movieId is 562.The rating is 3.0.The timestamp is 1166586512.
 Likes

----------------------------------------------------------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=2) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The userId is 15.The movieId is 574.The rating is 3.0.The timestamp is 1141391972.
 Likes

----------------------------------------------------------------------------------
The userId is 15.The movieId is 586.The rating is 2.0.The timestamp is 1093028377.
 Likes

----------------------------------------------------------------------------------


# *T-Few*

In [ ]:
#T-Few method for determining an answer
labeled_df = pd.DataFrame(labeled_examples)
labeled_df.rename(columns={0: 'Sentence', 1: 'Label', 2: 'drop'}, inplace=True)
labeled_df = labeled_df.drop('drop',axis=1)
unlabeled_df = df
print(labeled_df)
print(unlabeled_df)

text_column = 0
label_column = 1

tokenizer = AutoTokenizer.from_pretrained("bigscience/mt0-large")

def preprocess_function(examples):
    model_inputs = tokenizer(examples[text_column], max_length=128, padding="max_length", truncation=True)
    labels = tokenizer(examples[label_column], max_length=3, padding="max_length", truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

labeled_df = labeled_df.map(preprocess_function)

train_df = labeled_df["Sentence"]
eval_df = labeled_df["Label"]

train_dataloader = DataLoader(train_df, shuffle=True, collate_fn=default_data_collator, batch_size=8, pin_memory=True)
eval_dataloader = DataLoader(eval_df, collate_fn=default_data_collator, batch_size=8, pin_memory=True)

model = AutoModelForSeq2SeqLM.from_pretrained("bigscience/mt0-large")

peft_config = IA3Config(task_type="SEQ_2_SEQ_LM")
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

                                              Sentence     Label
0    The userId is 1.The movieId is 31.The rating i...  Dislikes
1    The userId is 1.The movieId is 1061.The rating...  Dislikes
2    The userId is 1.The movieId is 1172.The rating...     Likes
3    The userId is 1.The movieId is 1287.The rating...  Dislikes
4    The userId is 1.The movieId is 1339.The rating...     Likes
..                                                 ...       ...
995  The userId is 15.The movieId is 6380.The ratin...  Dislikes
996  The userId is 15.The movieId is 6433.The ratin...  Dislikes
997  The userId is 15.The movieId is 6464.The ratin...  Dislikes
998  The userId is 15.The movieId is 6503.The ratin...  Dislikes
999  The userId is 15.The movieId is 6537.The ratin...  Dislikes

[1000 rows x 2 columns]
       userId  movieId  rating   timestamp  \
0          15       34     3.0   997938310   
1          15       36     1.0  1033345287   
2          15       50     5.0   997938500   
3          

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 282,624 || all params: 1,229,863,936 || trainable%: 0.0230


In [ ]:
num_epochs = 3

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=(len(train_dataloader) * num_epochs),
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for step, batch in enumerate(tqdm(train_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.detach().float()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

    model.eval()
    eval_loss = 0
    eval_preds = []
    for step, batch in enumerate(tqdm(eval_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        loss = outputs.loss
        eval_loss += loss.detach().float()
        eval_preds.extend(
            tokenizer.batch_decode(torch.argmax(outputs.logits, -1).detach().cpu().numpy(), skip_special_tokens=True)
        )

    eval_epoch_loss = eval_loss / len(eval_dataloader)
    eval_ppl = torch.exp(eval_epoch_loss)
    train_epoch_loss = total_loss / len(train_dataloader)
    train_ppl = torch.exp(train_epoch_loss)
    print(f"{epoch=}: {train_ppl=} {train_epoch_loss=} {eval_ppl=} {eval_epoch_loss=}")

model.save_pretrained("./my_ia3_model")
tokenizer.save_pretrained("./my_ia3_model")

100%|██████████| 125/125 [00:40<00:00,  3.11it/s]


epoch=0: train_ppl=tensor(351691.8438, device='cuda:0') train_epoch_loss=tensor(12.7705, device='cuda:0') eval_ppl=tensor(61.5819, device='cuda:0') eval_epoch_loss=tensor(4.1204, device='cuda:0')


100%|██████████| 125/125 [00:39<00:00,  3.14it/s]


epoch=1: train_ppl=tensor(39787.1055, device='cuda:0') train_epoch_loss=tensor(10.5913, device='cuda:0') eval_ppl=tensor(52.2267, device='cuda:0') eval_epoch_loss=tensor(3.9556, device='cuda:0')


100%|██████████| 125/125 [00:39<00:00,  3.15it/s]


epoch=2: train_ppl=tensor(14866.0498, device='cuda:0') train_epoch_loss=tensor(9.6068, device='cuda:0') eval_ppl=tensor(34.9655, device='cuda:0') eval_epoch_loss=tensor(3.5544, device='cuda:0')


('./my_ia3_model/tokenizer_config.json', './my_ia3_model/tokenizer.json')

In [ ]:
#load model and set it to evaluation mode
#evaluating with the base model did not yield better results
base_model = AutoModelForSeq2SeqLM.from_pretrained("bigscience/mt0-large")
model = PeftModel.from_pretrained(base_model, "./my_ia3_model")
model.to(device)
model.eval()

#test for 100 examples
for i in range(100):
  input = unlabeled_df.iloc[i].values[4] + f" Question: Do you think the user Likes or Dislikes the movie? Answer: "
  print(input)
  input = tokenizer(input, return_tensors="pt").to(device)

  with torch.no_grad():
      input = {k: v.to(device) for k, v in input.items()}
      outputs = model.generate(input_ids=input["input_ids"], max_new_tokens=10)
      prediction = tokenizer.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens=True)
      print(prediction)

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


The userId is 15.The movieId is 34.The rating is 3.0.The timestamp is 997938310. Question: Do you think the user Likes or Dislikes the movie? Answer: 
['Dislikes']
The userId is 15.The movieId is 36.The rating is 1.0.The timestamp is 1033345287. Question: Do you think the user Likes or Dislikes the movie? Answer: 
['Dislikes']
The userId is 15.The movieId is 50.The rating is 5.0.The timestamp is 997938500. Question: Do you think the user Likes or Dislikes the movie? Answer: 
['Dislikes']
The userId is 15.The movieId is 52.The rating is 2.5.The timestamp is 1093070271. Question: Do you think the user Likes or Dislikes the movie? Answer: 
['Dislikes']
The userId is 15.The movieId is 94.The rating is 3.0.The timestamp is 1166587063. Question: Do you think the user Likes or Dislikes the movie? Answer: 
['Dislikes']
The userId is 15.The movieId is 104.The rating is 1.0.The timestamp is 1093070113. Question: Do you think the user Likes or Dislikes the movie? Answer: 
['Dislikes']
The userId 

# *T-Few + Text Modification Part 1*

In [ ]:
#it's possible that the reason for the above results is because the sentences are too complex, so we'll simplify them and retrain it
for i in range(len(labeled_examples)):
  labeled_examples[i] = (f"The rating is: {labeled_examples[i][2]}/5.0",labeled_examples[i][1])

#modify the format of the sentences on the test dataframe as well
for j in range(len(df)):
  df.iloc[j,2] = f"The rating is: {float(df.iloc[j,2])}/5.0"

print(labeled_examples[0])
print(df)

/tmp/ipykernel_4908/1731647686.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The rating is: 3.0/5.0' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.iloc[j,2] = f"The rating is: {float(df.iloc[j,2])}/5.0"


('The rating is: 2.5/5.0', 'Dislikes')
       userId  movieId                  rating   timestamp  \
0          15       34  The rating is: 3.0/5.0   997938310   
1          15       36  The rating is: 1.0/5.0  1033345287   
2          15       50  The rating is: 5.0/5.0   997938500   
3          15       52  The rating is: 2.5/5.0  1093070271   
4          15       94  The rating is: 3.0/5.0  1166587063   
...       ...      ...                     ...         ...   
98999     671     6268  The rating is: 2.5/5.0  1065579370   
99000     671     6269  The rating is: 4.0/5.0  1065149201   
99001     671     6365  The rating is: 4.0/5.0  1070940363   
99002     671     6385  The rating is: 2.5/5.0  1070979663   
99003     671     6565  The rating is: 3.5/5.0  1074784724   

                                                Sentence  
0      The userId is 15.The movieId is 34.The rating ...  
1      The userId is 15.The movieId is 36.The rating ...  
2      The userId is 15.The movieId is 

In [ ]:
#same process as above
labeled_df = pd.DataFrame(labeled_examples)
labeled_df.rename(columns={0: 'Sentence', 1: 'Label'}, inplace=True)
unlabeled_df = df['rating']
print(labeled_df)
print(unlabeled_df)

text_column = 0
label_column = 1

tokenizer = AutoTokenizer.from_pretrained("bigscience/mt0-large")

def preprocess_function(examples):
    model_inputs = tokenizer(examples[text_column], max_length=128, padding="max_length", truncation=True)
    labels = tokenizer(examples[label_column], max_length=3, padding="max_length", truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

labeled_df = labeled_df.map(preprocess_function)

train_df = labeled_df["Sentence"]
eval_df = labeled_df["Label"]

train_dataloader = DataLoader(train_df, shuffle=True, collate_fn=default_data_collator, batch_size=8, pin_memory=True)
eval_dataloader = DataLoader(eval_df, collate_fn=default_data_collator, batch_size=8, pin_memory=True)

model = AutoModelForSeq2SeqLM.from_pretrained("bigscience/mt0-large")

peft_config = IA3Config(task_type="SEQ_2_SEQ_LM")
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

                   Sentence     Label
0    The rating is: 2.5/5.0  Dislikes
1    The rating is: 3.0/5.0  Dislikes
2    The rating is: 4.0/5.0     Likes
3    The rating is: 2.0/5.0  Dislikes
4    The rating is: 3.5/5.0     Likes
..                      ...       ...
995  The rating is: 3.0/5.0  Dislikes
996  The rating is: 3.0/5.0  Dislikes
997  The rating is: 2.5/5.0  Dislikes
998  The rating is: 2.5/5.0  Dislikes
999  The rating is: 2.5/5.0  Dislikes

[1000 rows x 2 columns]
0        The rating is: 3.0/5.0
1        The rating is: 1.0/5.0
2        The rating is: 5.0/5.0
3        The rating is: 2.5/5.0
4        The rating is: 3.0/5.0
                  ...          
98999    The rating is: 2.5/5.0
99000    The rating is: 4.0/5.0
99001    The rating is: 4.0/5.0
99002    The rating is: 2.5/5.0
99003    The rating is: 3.5/5.0
Name: rating, Length: 99004, dtype: object


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/800 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/430 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 282,624 || all params: 1,229,863,936 || trainable%: 0.0230


In [ ]:
num_epochs = 3

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=(len(train_dataloader) * num_epochs),
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for step, batch in enumerate(tqdm(train_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.detach().float()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

    model.eval()
    eval_loss = 0
    eval_preds = []
    for step, batch in enumerate(tqdm(eval_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        loss = outputs.loss
        eval_loss += loss.detach().float()
        eval_preds.extend(
            tokenizer.batch_decode(torch.argmax(outputs.logits, -1).detach().cpu().numpy(), skip_special_tokens=True)
        )

    eval_epoch_loss = eval_loss / len(eval_dataloader)
    eval_ppl = torch.exp(eval_epoch_loss)
    train_epoch_loss = total_loss / len(train_dataloader)
    train_ppl = torch.exp(train_epoch_loss)
    print(f"{epoch=}: {train_ppl=} {train_epoch_loss=} {eval_ppl=} {eval_epoch_loss=}")

model.save_pretrained("./my_ia3_model")
tokenizer.save_pretrained("./my_ia3_model")

100%|██████████| 125/125 [00:38<00:00,  3.25it/s]


epoch=0: train_ppl=tensor(385460.5312, device='cuda:0') train_epoch_loss=tensor(12.8622, device='cuda:0') eval_ppl=tensor(100.4286, device='cuda:0') eval_epoch_loss=tensor(4.6094, device='cuda:0')


100%|██████████| 125/125 [00:37<00:00,  3.33it/s]


epoch=1: train_ppl=tensor(47855.8438, device='cuda:0') train_epoch_loss=tensor(10.7759, device='cuda:0') eval_ppl=tensor(84.8888, device='cuda:0') eval_epoch_loss=tensor(4.4413, device='cuda:0')


100%|██████████| 125/125 [00:37<00:00,  3.32it/s]


epoch=2: train_ppl=tensor(18720.3789, device='cuda:0') train_epoch_loss=tensor(9.8374, device='cuda:0') eval_ppl=tensor(82.8421, device='cuda:0') eval_epoch_loss=tensor(4.4169, device='cuda:0')


('./my_ia3_model/tokenizer_config.json', './my_ia3_model/tokenizer.json')

In [ ]:
base_model = AutoModelForSeq2SeqLM.from_pretrained("bigscience/mt0-large")
model = PeftModel.from_pretrained(base_model, "./my_ia3_model")
model.to(device)
model.eval()

for i in range(100):
  input = unlabeled_df.iloc[i] + f" Does the user like or dislike this movie?"
  print(input)
  input = tokenizer(input, return_tensors="pt").to(device)

  with torch.no_grad():
      input = {k: v.to(device) for k, v in input.items()}
      outputs = model.generate(input_ids=input["input_ids"], max_new_tokens=5)
      prediction = tokenizer.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens=True)
      print(prediction)

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


The rating is: 3.0/5.0 Does the user like or dislike this movie?
['like']
The rating is: 1.0/5.0 Does the user like or dislike this movie?
['negative']
The rating is: 5.0/5.0 Does the user like or dislike this movie?
['positive']
The rating is: 2.5/5.0 Does the user like or dislike this movie?
['negative']
The rating is: 3.0/5.0 Does the user like or dislike this movie?
['like']
The rating is: 1.0/5.0 Does the user like or dislike this movie?
['negative']
The rating is: 4.0/5.0 Does the user like or dislike this movie?
['positive']
The rating is: 3.5/5.0 Does the user like or dislike this movie?
['3']
The rating is: 2.0/5.0 Does the user like or dislike this movie?
['positive']
The rating is: 3.0/5.0 Does the user like or dislike this movie?
['like']
The rating is: 4.0/5.0 Does the user like or dislike this movie?
['positive']
The rating is: 3.0/5.0 Does the user like or dislike this movie?
['like']
The rating is: 2.0/5.0 Does the user like or dislike this movie?
['positive']
The ratin

# *T-Few + Text Modification Part 2*

In [ ]:
labeled_df = pd.DataFrame(labeled_examples)
labeled_df.rename(columns={0: 'Sentence', 1: 'Label'}, inplace=True)
unlabeled_df = df
print(labeled_df)
print(unlabeled_df)

text_column = "Sentence"
label_column = "Label"

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
tokenizer.pad_token = tokenizer.eos_token

def preprocess_function(examples):
    prompt = f"Data: {examples[text_column]} Verdict:"
    answer = f" {examples[label_column]}"

    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(answer, add_special_tokens=False)["input_ids"] + [tokenizer.eos_token_id]

    input_ids = prompt_ids + answer_ids

    labels = [-100] * len(prompt_ids) + answer_ids

    padding_len = 128 - len(input_ids)
    input_ids += [tokenizer.pad_token_id] * padding_len
    labels += [-100] * padding_len

    return {
        "input_ids": torch.tensor(input_ids[:128]),
        "labels": torch.tensor(labels[:128]),
        "attention_mask": torch.tensor(([1] * len(input_ids[:128])))
    }

full_dataset = Dataset.from_pandas(labeled_df)
split_dataset = full_dataset.train_test_split(test_size=0.2)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

tokenized_train = train_dataset.map(preprocess_function, batched=False)
tokenized_eval = eval_dataset.map(preprocess_function, batched=False)

tokenized_train = tokenized_train.remove_columns(["Sentence", "Label"])
tokenized_eval = tokenized_eval.remove_columns(["Sentence", "Label"])

train_dataloader = DataLoader(tokenized_train, shuffle=True, collate_fn=default_data_collator, batch_size=8)
eval_dataloader = DataLoader(tokenized_eval, collate_fn=default_data_collator, batch_size=8)

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

peft_config = IA3Config(task_type="CAUSAL_LM")
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

                   Sentence     Label
0    The rating is: 2.5/5.0  Dislikes
1    The rating is: 3.0/5.0  Dislikes
2    The rating is: 4.0/5.0     Likes
3    The rating is: 2.0/5.0  Dislikes
4    The rating is: 3.5/5.0     Likes
..                      ...       ...
995  The rating is: 3.0/5.0  Dislikes
996  The rating is: 3.0/5.0  Dislikes
997  The rating is: 2.5/5.0  Dislikes
998  The rating is: 2.5/5.0  Dislikes
999  The rating is: 2.5/5.0  Dislikes

[1000 rows x 2 columns]
0        The rating is: 3.0/5.0
1        The rating is: 1.0/5.0
2        The rating is: 5.0/5.0
3        The rating is: 2.5/5.0
4        The rating is: 3.0/5.0
                  ...          
98999    The rating is: 2.5/5.0
99000    The rating is: 4.0/5.0
99001    The rating is: 4.0/5.0
99002    The rating is: 2.5/5.0
99003    The rating is: 3.5/5.0
Name: rating, Length: 99004, dtype: object


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 301,056 || all params: 1,544,015,360 || trainable%: 0.0195


In [ ]:
num_epochs = 3

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=(len(train_dataloader) * num_epochs),
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for step, batch in enumerate(tqdm(train_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.detach().float()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

    model.eval()
    eval_loss = 0
    eval_preds = []
    for step, batch in enumerate(tqdm(eval_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        loss = outputs.loss
        eval_loss += loss.detach().float()
        eval_preds.extend(
            tokenizer.batch_decode(torch.argmax(outputs.logits, -1).detach().cpu().numpy(), skip_special_tokens=True)
        )

    eval_epoch_loss = eval_loss / len(eval_dataloader)
    eval_ppl = torch.exp(eval_epoch_loss)
    train_epoch_loss = total_loss / len(train_dataloader)
    train_ppl = torch.exp(train_epoch_loss)
    print(f"{epoch=}: {train_ppl=} {train_epoch_loss=} {eval_ppl=} {eval_epoch_loss=}")

model.save_pretrained("./my_ia3_model")
tokenizer.save_pretrained("./my_ia3_model")

100%|██████████| 25/25 [00:36<00:00,  1.44s/it]


epoch=0: train_ppl=tensor(10367.1855, device='cuda:0') train_epoch_loss=tensor(9.2464, device='cuda:0') eval_ppl=tensor(1.4834, device='cuda:0') eval_epoch_loss=tensor(0.3943, device='cuda:0')


100%|██████████| 25/25 [00:36<00:00,  1.44s/it]


epoch=1: train_ppl=tensor(1.3081, device='cuda:0') train_epoch_loss=tensor(0.2686, device='cuda:0') eval_ppl=tensor(1.2152, device='cuda:0') eval_epoch_loss=tensor(0.1949, device='cuda:0')


100%|██████████| 25/25 [00:36<00:00,  1.44s/it]


epoch=2: train_ppl=tensor(1.1933, device='cuda:0') train_epoch_loss=tensor(0.1767, device='cuda:0') eval_ppl=tensor(1.1839, device='cuda:0') eval_epoch_loss=tensor(0.1688, device='cuda:0')


('./my_ia3_model/tokenizer_config.json',
 './my_ia3_model/chat_template.jinja',
 './my_ia3_model/tokenizer.json')

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
model = PeftModel.from_pretrained(base_model, "./my_ia3_model")
model.to(device)
model.eval()

for i in range(100):
  input = unlabeled_df.iloc[i] + f" Does the user like or dislike this movie?"
  input = tokenizer(input, return_tensors="pt").to(device)

  with torch.no_grad():
      input = {k: v.to(device) for k, v in input.items()}
      outputs = model.generate(input_ids=input["input_ids"], max_new_tokens=2)
      prediction = tokenizer.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens=True)
      print(prediction)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

['The rating is: 3.0/5.0 Does the user like or dislike this movie? Like\n']
['The rating is: 1.0/5.0 Does the user like or dislike this movie? Dislike']
['The rating is: 5.0/5.0 Does the user like or dislike this movie? Like\n\n']
['The rating is: 2.5/5.0 Does the user like or dislike this movie? Dislikes']
['The rating is: 3.0/5.0 Does the user like or dislike this movie? Like\n']
['The rating is: 1.0/5.0 Does the user like or dislike this movie? Dislike']
['The rating is: 4.0/5.0 Does the user like or dislike this movie? Like\n']
['The rating is: 3.5/5.0 Does the user like or dislike this movie? Dislikes']
['The rating is: 2.0/5.0 Does the user like or dislike this movie? Likes it']
['The rating is: 3.0/5.0 Does the user like or dislike this movie? Like\n']
['The rating is: 4.0/5.0 Does the user like or dislike this movie? Like\n']
['The rating is: 3.0/5.0 Does the user like or dislike this movie? Likes it']
["The rating is: 2.0/5.0 Does the user like or dislike this movie? It's"]
['